In [1]:
#http://127.0.0.1:8888/?token=641ff58c730676d8f40d01ea1af56b3dc8e8be248d9aeb4a
import os
import re
import operator
import matplotlib.pyplot as plt
import warnings
import gensim
import numpy as np
warnings.filterwarnings('ignore')  # Let's not pay heed to them right now
import nltk
from gensim.models import CoherenceModel, LdaModel, LsiModel, HdpModel
from gensim.models.wrappers import LdaMallet
from gensim.corpora import Dictionary
from pprint import pprint
import json
import pyLDAvis.gensim
import datetime
from nltk.stem.lancaster import LancasterStemmer
from nltk.corpus import stopwords
from nltk.stem.porter import *
import numpy as np
import string
%matplotlib inline
import pandas as pd
from nltk import word_tokenize
from datetime import datetime
import datetime as dt


In [2]:
# import file to calculate LDA
# REDDIT
filename = 'IllegalLifeProTips_messages_08042020_illigal_movies_streaming.csv'
df = pd.read_csv(os.path.join(os.getcwd(), 'reddit_data', 'clean_data', filename))
print(len(df))

86159


In [3]:
df.columns

Index(['Unnamed: 0', 'subreddit', 'body', 'created_utc', 'author'], dtype='object')

In [4]:
def from_unix_to_datestamp(date_time_stamp):
    """
    Transform from unix to datestamp date
    :param date_time_stamp: single unix format date
    :return: datestamp date
    """
    date_time_stamp = int(date_time_stamp)
    return datetime.utcfromtimestamp(date_time_stamp).strftime('%Y-%m-%d %H:%M:%S')


In [5]:
# # datestamp
# df['date'] = [from_unix_to_datestamp(ts) for ts in df['created_utc'].tolist()]
# df['date'] = df['date'].apply(lambda x: dt.datetime.strftime(x, '%Y-%m-%d'))
# df = df.set_index('date')

In [6]:
# Select only data desidered:
# df = df[df.author == 'mindy2000']
#df1 = df[df.subreddit == 'Futurology']
print(len(df))

86159


In [7]:
def build_texts(fname):
    """
    Function to build tokenized texts from file
    fname: File to be read
    Returns: yields preprocessed line
    """
    
    for line in fname:
        yield gensim.utils.simple_preprocess(line, deacc=True, min_len=3)
            

def clean_text(text):

    stop_words = set(stopwords.words('english'))
    #porter = PorterStemmer()
    #lemman = LancasterStemmer()
    translator = str.maketrans('', '', string.punctuation)
    text = re.sub(r'http\S+', '', text)
    text = text.translate(translator)

    # POS
    list_tags = ['NN', 'NNP', 'JJ', 'FW', 'NNS', 'NNPS', 'JJ', 'JJR', 'JJS']
    text = tagging(text, allowed_postags=list_tags)

    return ' '.join([w for w in text.split() if w not in stop_words])
    #return ' '.join([lemman.stem(porter.stem(w)) for w in text.split() if w not in stop_words])

    # Tagging

def tagging(text, allowed_postags):
    """
    Remove some not selected POS
    :param text: string text
    :param allowed_postags: POS selected as a list
    :return: tagged words
    """
    text = word_tokenize(text)
    text = ' '.join([item[0] for item in nltk.pos_tag(text) if item[1] in allowed_postags])
    return text


In [8]:
# Clean text in dataframe
cleaned_data = [clean_text(d) for d in df['body'].tolist()]
train_texts = list(build_texts(cleaned_data))
print(f'Number of messages --> {len(train_texts)}')


Number of messages --> 86159


In [9]:
print(train_texts[:3])

[['get', 'inject', 'insulin', 'hair', 'follicle', 'head', 'get', 'rid', 'evidence'], ['order', 'signature', 'delivery'], ['marched', 'madness', 'press', 'box', 'memphis', 'confidence']]


In [10]:
dictionary = Dictionary(train_texts)
corpus = [dictionary.doc2bow(text) for text in train_texts]

In [11]:
import random
random.seed(113)
ldamodel = LdaModel(corpus=corpus, num_topics=4, id2word=dictionary)

In [12]:

def topic_number(model_corpus):
    try:
        val = {}
        for t, v in model_corpus:
            val[t] = v
        topic = max(val, key=val.get) if len(val) > 0 else None

        weight = val[topic]
        return weight, topic
    except:
#         print('MMmmm')
        return 0, 0

In [13]:
print(df.columns)

Index(['Unnamed: 0', 'subreddit', 'body', 'created_utc', 'author'], dtype='object')


In [14]:
ldamodel.print_topic(0, topn=30)

'0.088*"ilpt" + 0.084*"rule" + 0.054*"please" + 0.049*"questions" + 0.048*"bot" + 0.047*"hello" + 0.047*"action" + 0.046*"concerns" + 0.046*"moderators" + 0.044*"request" + 0.043*"start" + 0.043*"title" + 0.042*"following" + 0.042*"submission" + 0.011*"beach" + 0.011*"link" + 0.008*"interested" + 0.008*"dna" + 0.007*"did" + 0.006*"google" + 0.005*"arrive" + 0.005*"thank" + 0.004*"description" + 0.004*"morons" + 0.004*"geo" + 0.002*"nices" + 0.002*"pls" + 0.002*"gym" + 0.002*"mother" + 0.002*"boo"'

In [15]:
ldamodel[corpus[0]]

[(0, 0.12678714), (1, 0.23538771), (2, 0.61058235), (3, 0.027242752)]

In [16]:
coherence_model_nmf = CoherenceModel(model=ldamodel, texts=train_texts, dictionary=dictionary,
                                             coherence='c_v')
coherence_lda = coherence_model_nmf.get_coherence()
print('\nCoherence Score: ', coherence_lda)


Coherence Score:  0.5421729732339914


In [17]:
pyLDAvis.enable_notebook()

In [18]:
data = pyLDAvis.gensim.prepare(ldamodel, corpus, dictionary, n_jobs=1)

In [19]:
data

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
1      0.148330 -0.090772       1        1  32.218136
3      0.145141 -0.129723       2        1  29.652115
2      0.134297  0.225264       3        1  24.701128
0     -0.427768 -0.004769       4        1  13.428615, topic_info=      Category         Freq       Term        Total  loglift  logprob
104    Default  7159.000000       ilpt  7159.000000  30.0000  30.0000
109    Default  6884.000000       rule  6884.000000  29.0000  29.0000
106    Default  4417.000000     please  4417.000000  28.0000  28.0000
107    Default  3980.000000  questions  3980.000000  27.0000  27.0000
100    Default  3956.000000        bot  3956.000000  26.0000  26.0000
...        ...          ...        ...          ...      ...      ...
227     Topic4   619.722229        dna   761.445679   1.8018  -4.8810
27894   Topic4   173.801315      nices   189.626587   1.9206  -6.1523
4365    Topic4   140.877914        gym   151.334961   1.9362  -6.3624
699     Topic4   377.733398      thank   664.851685   1.4424  -5.3761
229     Topic4   506.877167     google  1078.282227   1.2529  -5.0820

[287 rows x 6 columns], token_table=      Topic      Freq     Term
term                          
544       1  0.672450     able
544       2  0.202295     able
544       3  0.124735     able
720       1  0.985239  account
720       2  0.001250  account
...     ...       ...      ...
97        3  0.002854     yeah
166       1  0.276921    youre
166       2  0.246651    youre
166       3  0.476111    youre
364       1  0.995666  youtube

[485 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[2, 4, 3, 1])

In [20]:
pyLDAvis.save_html(data,'Topic_Modelling_ILPT.html')